In [11]:
# --- Cell 1: Imports and Setup ---
import warnings
import os
import numpy as np
import pandas as pd
import xgboost as xgb
import catboost as cb
import optuna
from scipy.optimize import minimize, minimize_scalar
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold

# --- Global Constants ---
DATA_PATH = './'
PREDS_PATH = './model_predictions/'
RANDOM_STATE = 42
N_SPLITS = 5

# --- Winkler Score Helper Function ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = np.mean(width + penalty_lower + penalty_upper)
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return score, coverage, np.mean(width)
    return score

warnings.filterwarnings('ignore')
print("Libraries and helper functions loaded successfully.")

Libraries and helper functions loaded successfully.


In [12]:
# --- Cell 2 (Corrected): Load Data and Create RICH Meta-Feature Set ---
print("--- Loading all pre-trained base model predictions and raw data ---")
try:
    # ... (Loading predictions and y_true is the same as before) ...
    oof_xgb_preds = np.load(f'{PREDS_PATH}oof_xgb_preds.npy')
    test_xgb_preds = np.load(f'{PREDS_PATH}test_xgb_preds.npy')
    oof_cb_preds = np.load(f'{PREDS_PATH}oof_cb_preds.npy')
    test_cb_preds = np.load(f'{PREDS_PATH}test_cb_preds.npy')
    oof_nn_preds = np.load(f'{PREDS_PATH}oof_nn_preds.npy')
    test_nn_preds = np.load(f'{PREDS_PATH}test_nn_preds.npy')
    y_true = pd.read_csv(DATA_PATH + 'dataset.csv')['sale_price']
    df_train_raw = pd.read_csv(DATA_PATH + 'dataset.csv')
    df_test_raw = pd.read_csv(DATA_PATH + 'test.csv')
    print("Base predictions and raw data loaded.")

    # --- 1. Create the Super-Ensemble Mean Prediction (same as before) ---
    # ... (Clipping and optimizing for mean blend weights is the same) ...
    oof_xgb_preds_orig, oof_cb_preds_orig, oof_nn_preds_orig = np.expm1(oof_xgb_preds), np.expm1(oof_cb_preds), np.expm1(oof_nn_preds)
    max_reasonable_price = y_true.max() * 1.1
    oof_xgb_preds_orig = np.nan_to_num(oof_xgb_preds_orig, posinf=max_reasonable_price)
    oof_cb_preds_orig = np.nan_to_num(oof_cb_preds_orig, posinf=max_reasonable_price)
    oof_nn_preds_orig = np.nan_to_num(oof_nn_preds_orig, posinf=max_reasonable_price)
    oof_preds_stack_orig = np.vstack([oof_xgb_preds_orig, oof_cb_preds_orig, oof_nn_preds_orig]).T
    def get_ensemble_rmse(weights):
        final_prediction = np.dot(oof_preds_stack_orig, weights)
        return np.sqrt(mean_squared_error(y_true, final_prediction))
    result_mean = minimize(get_ensemble_rmse, [1/3]*3, method='SLSQP', bounds=[(0,1)]*3, constraints=({'type': 'eq', 'fun': lambda w: 1 - sum(w)}))
    best_mean_weights = result_mean.x
    oof_preds_stack_log = np.vstack([oof_xgb_preds, oof_cb_preds, oof_nn_preds]).T
    test_preds_stack_log = np.vstack([test_xgb_preds, test_cb_preds, test_nn_preds]).T
    oof_ensemble_mean_log = np.dot(oof_preds_stack_log, best_mean_weights)
    test_ensemble_mean_log = np.dot(test_preds_stack_log, best_mean_weights)
    print("Super-ensemble mean predictions recreated.")

    # --- 2. Perform FULL Feature Engineering ---
    print("Performing full feature engineering...")
    df_train_fe = df_train_raw.drop(columns=['id', 'sale_price'])
    df_test_fe = df_test_raw.drop(columns=['id'])
    all_data = pd.concat([df_train_fe, df_test_fe], axis=0).reset_index(drop=True)
    date_column = 'sale_date'
    all_data[date_column] = pd.to_datetime(all_data[date_column])
    all_data['sale_year'] = all_data[date_column].dt.year
    all_data['sale_month'] = all_data[date_column].dt.month
    all_data['property_age'] = all_data['sale_year'] - all_data['year_built']
    all_data['property_age'] = np.maximum(0, all_data['property_age'])
    all_data['sale_nbr'].fillna(all_data['sale_nbr'].median(), inplace=True)
    all_data['subdivision'].fillna('None', inplace=True)
    all_data['submarket'].fillna(all_data['submarket'].mode()[0], inplace=True)
    all_data.drop(columns=[date_column, 'year_built'], inplace=True)
    categorical_cols = all_data.select_dtypes(include='object').columns
    all_data = pd.get_dummies(all_data, columns=categorical_cols, dummy_na=False)
    X_train_fe, X_test_fe = all_data.iloc[:len(df_train_raw)], all_data.iloc[len(df_train_raw):]
    X_train_fe, X_test_fe = X_train_fe.align(X_test_fe, join='inner', axis=1)
    print("Full feature engineering complete.")

    # --- 3. Combine into Final RICH Meta-Feature Sets ---
    base_model_preds_train = pd.DataFrame({
        'xgb_pred_log': oof_xgb_preds, 'cb_pred_log': oof_cb_preds, 'nn_pred_log': oof_nn_preds,
        'ensemble_mean_log': oof_ensemble_mean_log
    })
    base_model_preds_test = pd.DataFrame({
        'xgb_pred_log': test_xgb_preds, 'cb_pred_log': test_cb_preds, 'nn_pred_log': test_nn_preds,
        'ensemble_mean_log': test_ensemble_mean_log
    })

    X_meta_train = pd.concat([base_model_preds_train, X_train_fe.reset_index(drop=True)], axis=1)
    X_meta_test = pd.concat([base_model_preds_test, X_test_fe.reset_index(drop=True)], axis=1)
    
    # Ensure column names are strings for XGBoost compatibility
    X_meta_train.columns = X_meta_train.columns.astype(str)
    X_meta_test.columns = X_meta_test.columns.astype(str)

    print(f"Final RICH meta-feature sets created.")
    print(f"X_meta_train shape: {X_meta_train.shape}")
    print(f"X_meta_test shape: {X_meta_test.shape}")

except FileNotFoundError as e:
    print(f"\nFATAL ERROR: Could not find a required file. {e}")
    raise

--- Loading all pre-trained base model predictions and raw data ---
Base predictions and raw data loaded.
Super-ensemble mean predictions recreated.
Performing full feature engineering...
Full feature engineering complete.
Final RICH meta-feature sets created.
X_meta_train shape: (200000, 11490)
X_meta_test shape: (200000, 11490)


In [ ]:
# --- Cell 3: Targeted Tuning for XGBoost Meta-Model (Definitive Fix) ---
import optuna
import xgboost as xgb # Use the core library import

print("\n--- Tuning XGBoost Meta-Model with Optuna and ROBUST CV (Native API) ---")

X = X_meta_train
y = y_true

def objective_xgb(trial):
    # Same targeted search space as before
    params = {
        'objective': 'reg:quantileerror',
        'eval_metric': 'rmse',
        'n_jobs': -1,
        'tree_method': 'hist',
        'seed': RANDOM_STATE,
        'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.04),
        'max_depth': trial.suggest_int('max_depth', 5, 8),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),
        'lambda': trial.suggest_float('lambda', 1.0, 20.0),
        'alpha': trial.suggest_float('alpha', 1.0, 20.0),
    }
    
    fold_scores = []
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # *** THE FIX: Convert data to XGBoost's native DMatrix format ***
        dtrain_lower = xgb.DMatrix(X_train, label=y_train)
        dval_lower = xgb.DMatrix(X_val, label=y_val)
        
        dtrain_upper = xgb.DMatrix(X_train, label=y_train)
        dval_upper = xgb.DMatrix(X_val, label=y_val)

        # --- Train Lower Bound Model using xgb.train ---
        params_lower = params.copy()
        params_lower['quantile_alpha'] = 0.05
        model_lower = xgb.train(
            params=params_lower,
            dtrain=dtrain_lower,
            num_boost_round=2000, # Max rounds
            evals=[(dval_lower, 'eval')],
            early_stopping_rounds=40, # This is the correct parameter for xgb.train
            verbose_eval=False
        )
        
        # --- Train Upper Bound Model using xgb.train ---
        params_upper = params.copy()
        params_upper['quantile_alpha'] = 0.95
        model_upper = xgb.train(
            params=params_upper,
            dtrain=dtrain_upper,
            num_boost_round=2000,
            evals=[(dval_upper, 'eval')],
            early_stopping_rounds=40,
            verbose_eval=False
        )
        
        # Predict using the trained models
        lower_preds = model_lower.predict(dval_lower, iteration_range=(0, model_lower.best_iteration))
        upper_preds = model_upper.predict(dval_upper, iteration_range=(0, model_upper.best_iteration))

        lower_preds = np.minimum(lower_preds, upper_preds)
        score = winkler_score(y_val, lower_preds, upper_preds)
        fold_scores.append(score)
        
    return np.mean(fold_scores)

# --- Run the study ---
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=25)

best_params_xgb = study_xgb.best_params
print(f"\n--- XGBoost Optuna Study Complete ---")
print(f"Best XGBoost parameters found: {best_params_xgb}")
print(f"Best CV Winkler Score for XGBoost: {study_xgb.best_value:,.2f}")

[I 2025-07-20 20:50:07,278] A new study created in memory with name: no-name-a3bf9624-3628-4a2e-82fb-c57f9f133128



--- Tuning XGBoost Meta-Model with Optuna and ROBUST CV (Native API) ---


In [ ]:
# --- Cell 4: Targeted Tuning for CatBoost Meta-Model ---
print("\n--- Tuning CatBoost Meta-Model with Optuna and ROBUST CV (Targeted Search) ---")
def objective_cb(trial):
    # Search space is TIGHTLY centered around the best params from the PDF
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.035, 0.055), # Centered on 0.045
        'depth': trial.suggest_int('depth', 6, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 8.0, 12.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.4, 0.65),
        'random_strength': trial.suggest_float('random_strength', 0.15, 0.35),
    }
    cb_base_params = {
        'eval_metric': 'Quantile', 'iterations': 2500, 'random_seed': RANDOM_STATE,
        'verbose': 0, 'thread_count': -1, **params
    }
    fold_scores = []
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        cb_lower = cb.CatBoostRegressor(**cb_base_params, loss_function='Quantile:alpha=0.05')
        cb_lower.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=40, use_best_model=True)
        cb_upper = cb.CatBoostRegressor(**cb_base_params, loss_function='Quantile:alpha=0.95')
        cb_upper.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=40, use_best_model=True)
        lower_preds = cb_lower.predict(X_val)
        upper_preds = cb_upper.predict(X_val)
        lower_preds = np.minimum(lower_preds, upper_preds)
        score = winkler_score(y_val, lower_preds, upper_preds)
        fold_scores.append(score)
    return np.mean(fold_scores)

study_cb = optuna.create_study(direction='minimize')
study_cb.optimize(objective_cb, n_trials=25)
best_params_catboost = study_cb.best_params
print(f"\n--- CatBoost Optuna Study Complete ---")
print(f"Best CatBoost parameters found: {best_params_catboost}")
print(f"Best CV Winkler Score for CatBoost: {study_cb.best_value:,.2f}")

In [ ]:
# --- Cell 5: Final Model Training, Blending, and Submission ---
print("\n--- Training Final L1 Meta-Models and Creating Submission ---")
kf_final = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
# Create all OOF and test prediction arrays
oof_cb_lower, oof_cb_upper = np.zeros(len(X)), np.zeros(len(X))
test_cb_lower, test_cb_upper = np.zeros(len(X_meta_test)), np.zeros(len(X_meta_test))
oof_xgb_lower, oof_xgb_upper = np.zeros(len(X)), np.zeros(len(X))
test_xgb_lower, test_xgb_upper = np.zeros(len(X_meta_test)), np.zeros(len(X_meta_test))

# Final model parameters are the best ones found by Optuna
final_cb_params = {
    'eval_metric': 'Quantile', 'iterations': 5000, 'random_seed': RANDOM_STATE,
    'verbose': 0, 'thread_count': -1, **best_params_catboost
}
final_xgb_params = {
    'objective': 'reg:quantileerror', 'eval_metric': 'rmse', 'n_estimators': 5000,
    'n_jobs': -1, 'seed': RANDOM_STATE, 'tree_method': 'hist', **best_params_xgb
}

# --- Train Final Models with CV ---
for fold, (train_idx, val_idx) in enumerate(kf_final.split(X, y)):
    print(f"Training final models - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Train CatBoost
    cb_lower = cb.CatBoostRegressor(**final_cb_params, loss_function='Quantile:alpha=0.05')
    cb_lower.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, use_best_model=True)
    cb_upper = cb.CatBoostRegressor(**final_cb_params, loss_function='Quantile:alpha=0.95')
    cb_upper.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, use_best_model=True)
    oof_cb_lower[val_idx], oof_cb_upper[val_idx] = cb_lower.predict(X_val), cb_upper.predict(X_val)
    test_cb_lower += cb_lower.predict(X_meta_test) / N_SPLITS
    test_cb_upper += cb_upper.predict(X_meta_test) / N_SPLITS
    
    # Train XGBoost
    xgb_lower = xgb.XGBRegressor(**final_xgb_params, quantile_alpha=0.05)
    xgb_lower.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    xgb_upper = xgb.XGBRegressor(**final_xgb_params, quantile_alpha=0.95)
    xgb_upper.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    oof_xgb_lower[val_idx], oof_xgb_upper[val_idx] = xgb_lower.predict(X_val), xgb_upper.predict(X_val)
    test_xgb_lower += xgb_lower.predict(X_meta_test) / N_SPLITS
    test_xgb_upper += xgb_upper.predict(X_meta_test) / N_SPLITS

# --- Comprehensive Blending Analysis and Final Showdown ---
print("\n--- Final Showdown: Comparing All Strategies ---")
# Evaluate standalone models
score_xgb, cov_xgb, width_xgb = winkler_score(y_true, oof_xgb_lower, oof_xgb_upper, return_coverage=True)
score_cb, cov_cb, width_cb = winkler_score(y_true, oof_cb_lower, oof_cb_upper, return_coverage=True)

# Find optimal blend weight
def blend_objective(weight):
    w_cb = weight
    w_xgb = 1 - w_cb
    final_lower = w_cb * oof_cb_lower + w_xgb * oof_xgb_lower
    final_upper = w_cb * oof_cb_upper + w_xgb * oof_xgb_upper
    return winkler_score(y_true, final_lower, final_upper)

blend_result = minimize_scalar(blend_objective, bounds=(0, 1), method='bounded')
best_weight_cb = blend_result.x
best_weight_xgb = 1 - best_weight_cb
best_final_score = blend_result.fun

# --- Display Results ---
summary = pd.DataFrame({
    'Strategy': ['XGBoost Meta-Model ONLY', 'CatBoost Meta-Model ONLY', 'OPTIMIZER FINAL BLEND'],
    'Winkler Score': [score_xgb, score_cb, best_final_score],
    'Blend Weight for CatBoost': [0.0, 1.0, best_weight_cb]
}).sort_values('Winkler Score').reset_index(drop=True)

print(summary.to_string())



In [ ]:
# --- Create final submission based on the BEST strategy ---
final_test_lower = best_weight_cb * test_cb_lower + best_weight_xgb * test_xgb_lower
final_test_upper = best_weight_cb * test_cb_upper + best_weight_xgb * test_xgb_upper
final_test_lower = np.maximum(0, final_test_lower)
final_test_upper = np.maximum(final_test_lower, final_test_upper)

submission_df = pd.DataFrame({'id': df_test_raw['id'], 'pi_lower': final_test_lower, 'pi_upper': final_test_upper})
submission_filename = f'submission_targeted_robust_blend_{int(best_final_score)}.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n'{submission_filename}' created successfully!")
print("Submission file head:")
print(submission_df.head())